In [2]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score,classification_report
import torch
import requests
import re
import pandas as pd
import numpy as np

In [3]:
tokenizer = AutoTokenizer.from_pretrained("nlptown/bert-base-multilingual-uncased-sentiment")

model = AutoModelForSequenceClassification.from_pretrained("nlptown/bert-base-multilingual-uncased-sentiment")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [4]:
tokens = tokenizer.encode("It was okay!!", return_tensors="pt")

In [5]:
tokens

tensor([[  101, 10197, 10140, 44810, 10158,   106,   106,   102]])

In [6]:
result = model(tokens)

In [7]:
result.logits

tensor([[-1.6652,  0.7296,  2.7066,  0.4230, -1.9684]],
       grad_fn=<AddmmBackward0>)

In [8]:
int(torch.argmax(result.logits)) + 1

3

# Sentiment of the review as a number of stars
### 5 - Great
### 3 - Normal
### 1 - Bad

In [9]:
#Load the Amazon dataset
df = pd.read_csv("Amazon_Reviews_dataset.csv", engine="python",on_bad_lines="skip", encoding="latin-1")
df.head()

,Reviewer Name,Profile Link,Country,Review Count,Review Date,Rating,Review Title,Review Text,Date of Experience
0,Eugene ath,/users/66e8185ff1598352d6b3701a,US,1 review,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,A Store That Doesn't Want to Sell Anything,"I registered on the website, tried to order a ...","September 16, 2024"
1,Daniel ohalloran,/users/5d75e460200c1f6a6373648c,GB,9 reviews,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up andâ¦,Had multiple orders one turned up and driver h...,"September 16, 2024"
2,p fisher,/users/546cfcf1000064000197b88f,GB,90 reviews,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates,I informed these reprobates that I WOULD NOT B...,"September 16, 2024"
3,Greg Dunn,/users/62c35cdbacc0ea0012ccaffa,AU,5 reviews,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,Advertise one price then increase it on website,I have bought from Amazon before and no proble...,"September 17, 2024"
4,Sheila Hannah,/users/5ddbe429478d88251550610e,GB,8 reviews,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would,If I could give a lower rate I would! I cancel...,"September 16, 2024"


In [10]:
len(df)

1861

In [11]:
# Drop unwanted columns
df = df.drop(columns=["Profile Link", "Country", "Review Count", "Review Title", "Date of Experience"])

In [12]:
df

,Reviewer Name,Review Date,Rating,Review Text
0,Eugene ath,2024-09-16T13:44:26.000Z,Rated 1 out of 5 stars,"I registered on the website, tried to order a ..."
1,Daniel ohalloran,2024-09-16T18:26:46.000Z,Rated 1 out of 5 stars,Had multiple orders one turned up and driver h...
2,p fisher,2024-09-16T21:47:39.000Z,Rated 1 out of 5 stars,I informed these reprobates that I WOULD NOT B...
3,Greg Dunn,2024-09-17T07:15:49.000Z,Rated 1 out of 5 stars,I have bought from Amazon before and no proble...
4,Sheila Hannah,2024-09-16T18:37:17.000Z,Rated 1 out of 5 stars,If I could give a lower rate I would! I cancel...
...,...,...,...,...
1856,Mr Grover,2023-08-15T20:22:57.000Z,Rated 5 out of 5 stars,Nice easy refund and quick delivery To
1857,Chris Stanford,2023-08-15T14:58:04.000Z,Rated 5 out of 5 stars,Use it every day. Deliveries always perfect.
1858,mixx jonasz,2020-01-17T19:37:23.000Z,Rated 3 out of 5 stars,My experience with Amazon and I don't use them...
1859,Laura Fagg,2020-01-17T07:50:16.000Z,Rated 3 out of 5 stars,"Further to my last review, Amazon has made som..."


In [13]:
df.columns.tolist()

['Reviewer Name', 'Review Date', 'Rating', 'Review Text']

In [14]:
# Extract number and handle NaN values
df["Rating"] = df["Rating"].str.extract(r"(\d)").squeeze()
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
df = df.dropna(subset=["Rating"])  # drop rows with missing ratings
df["Rating"] = df["Rating"].astype(int)

print(df["Rating"].value_counts().sort_index())

Rating
1     98
2    266
3    550
4    425
5    522
Name: count, dtype: int64


In [15]:
df

,Reviewer Name,Review Date,Rating,Review Text
0,Eugene ath,2024-09-16T13:44:26.000Z,1,"I registered on the website, tried to order a ..."
1,Daniel ohalloran,2024-09-16T18:26:46.000Z,1,Had multiple orders one turned up and driver h...
2,p fisher,2024-09-16T21:47:39.000Z,1,I informed these reprobates that I WOULD NOT B...
3,Greg Dunn,2024-09-17T07:15:49.000Z,1,I have bought from Amazon before and no proble...
4,Sheila Hannah,2024-09-16T18:37:17.000Z,1,If I could give a lower rate I would! I cancel...
...,...,...,...,...
1856,Mr Grover,2023-08-15T20:22:57.000Z,5,Nice easy refund and quick delivery To
1857,Chris Stanford,2023-08-15T14:58:04.000Z,5,Use it every day. Deliveries always perfect.
1858,mixx jonasz,2020-01-17T19:37:23.000Z,3,My experience with Amazon and I don't use them...
1859,Laura Fagg,2020-01-17T07:50:16.000Z,3,"Further to my last review, Amazon has made som..."


In [16]:
def predict_sentiment(review_text):   
    """
    Predicts sentiment for a single review.
    Returns star rating (1-5) and label.
    """
    if not isinstance(review_text, str) or review_text.strip() == "":
        return None, "Unknown"
    
    # Truncate to 512 tokens (BERT's max limit)
    tokens = tokenizer.encode(
        review_text,              
        return_tensors="pt",
        truncation=True,
        max_length=512
    )
    
    with torch.no_grad():
        result = model(tokens)
    
    star_rating = int(torch.argmax(result.logits)) + 1
    
    if star_rating <= 2:
        label = "Negative"
    elif star_rating == 3:
        label = "Neutral"
    else:
        label = "Positive"
    
    return star_rating, label

In [17]:
REVIEW_COLUMN = "Review Text"
print(f"Running predictions on {len(df)} reviews...")

stars = []
labels = []

for i, review in enumerate(df[REVIEW_COLUMN]):
    star, label = predict_sentiment(review)
    stars.append(star)
    labels.append(label)
    
    # Progress update every 100 rows
    if (i + 1) % 100 == 0:
        print(f"  Processed {i+1}/{len(df)} reviews")

df["predicted_stars"] = stars
df["predicted_sentiment"] = labels

print("\n✅ Predictions complete!")
print(df[["Review Text"]].head(10))

Running predictions on 1861 reviews...
  Processed 100/1861 reviews
  Processed 200/1861 reviews
  Processed 300/1861 reviews
  Processed 400/1861 reviews
  Processed 500/1861 reviews
  Processed 600/1861 reviews
  Processed 700/1861 reviews
  Processed 800/1861 reviews
  Processed 900/1861 reviews
  Processed 1000/1861 reviews
  Processed 1100/1861 reviews
  Processed 1200/1861 reviews
  Processed 1300/1861 reviews
  Processed 1400/1861 reviews
  Processed 1500/1861 reviews
  Processed 1600/1861 reviews
  Processed 1700/1861 reviews
  Processed 1800/1861 reviews

✅ Predictions complete!
                                         Review Text
0  I registered on the website, tried to order a ...
1  Had multiple orders one turned up and driver h...
2  I informed these reprobates that I WOULD NOT B...
3  I have bought from Amazon before and no proble...
4  If I could give a lower rate I would! I cancel...
5  Terrible you get customer service reps that ar...
6  Amazon has a way of tainting a 

In [18]:
# Cleaner display
pd.set_option("display.max_colwidth", 50)  # trim long text
pd.set_option("display.width", 200)         # wider display

print(df[["Review Text", "predicted_stars", "predicted_sentiment"]].head(20))

                                          Review Text  predicted_stars predicted_sentiment
0   I registered on the website, tried to order a ...                1            Negative
1   Had multiple orders one turned up and driver h...                1            Negative
2   I informed these reprobates that I WOULD NOT B...                1            Negative
3   I have bought from Amazon before and no proble...                1            Negative
4   If I could give a lower rate I would! I cancel...                1            Negative
5   Terrible you get customer service reps that ar...                1            Negative
6   Amazon has a way of tainting a great product d...                1            Negative
7   I applied for a job with Amazon. I completed a...                1            Negative
8   I have no interest in signing up to Amazon Pri...                1            Negative
9   Sadly, Amazon no longer provides the quality c...                1            Negative

In [19]:
print("=== Sentiment Distribution ===")
print(df["predicted_sentiment"].value_counts())

print("\n=== Star Rating Distribution ===")
print(df["predicted_stars"].value_counts().sort_index())

=== Sentiment Distribution ===
predicted_sentiment
Negative    861
Positive    763
Neutral     237
Name: count, dtype: int64

=== Star Rating Distribution ===
predicted_stars
1    651
2    210
3    237
4    224
5    539
Name: count, dtype: int64


In [20]:
actual = df["Rating"].astype(int)
predicted = df["predicted_stars"]

print(f"Accuracy: {accuracy_score(actual, predicted)*100:.1f}%")
print("\n", classification_report(actual, predicted))

Accuracy: 39.8%

               precision    recall  f1-score   support

           1       0.15      0.97      0.25        98
           2       0.22      0.17      0.19       266
           3       0.57      0.25      0.35       550
           4       0.50      0.26      0.35       425
           5       0.65      0.67      0.66       522

    accuracy                           0.40      1861
   macro avg       0.42      0.47      0.36      1861
weighted avg       0.51      0.40      0.41      1861



In [32]:
# Show a single review and its sentiment

idx = 0

print("Review:")
print(df.loc[idx, "Review Text"])

print("\nPredicted Sentiment:")
print(df.loc[idx, "predicted_sentiment"])

print("\nPredicted Stars:")
print(df.loc[idx, "predicted_stars"])

Review:
I registered on the website, tried to order a laptop, entered all the details, but instead of charging me and sending the product, they froze my account, demanding various verification documents. I sent them over. They said they would review them within 24 hours. In reality, it's been a week, and no one can help or give any (truthful) estimate of when it will be resolved; they just tell me to 'wait.' I've never seen such a horrible marketplace in my life. I hope those who came up with this can't buy food in a store, receiving a 'document review request' that takes forever to process.

Predicted Sentiment:
Negative

Predicted Stars:
1


In [35]:
# Show a single review and its sentiment

idx = 130  # Change this to any review index

print("Review:")
print(df.loc[idx, "Review Text"])

print("\nPredicted Sentiment:")
print(df.loc[idx, "predicted_sentiment"])

print("\nPredicted Stars:")
print(df.loc[idx, "predicted_stars"])

Review:
Good company.lots of products.but would be better if they offered repeat customers a deal/break/discount.... i know people who have spent thousands of dollars and have not even been acknowledged....I just read that Amazon is approaching the "trillion" dollar mark in revenue.it's time to share the wealth people!!!!

Predicted Sentiment:
Neutral

Predicted Stars:
3


In [33]:
# Show a single review and its sentiment

idx = 230

print("Review:")
print(df.loc[idx, "Review Text"])

print("\nPredicted Sentiment:")
print(df.loc[idx, "predicted_sentiment"])

print("\nPredicted Stars:")
print(df.loc[idx, "predicted_stars"])

Review:
I have had excellent service with all goods orderd and received

Predicted Sentiment:
Positive

Predicted Stars:
5
